# RAG pipelines - Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/00/9ynj9xts727dx08n754vj79h0000gn/T/ipykernel_17868/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("/Users/nithiii/Projects/data/pdf_files")


Found 1 PDF files to process

Processing: SampleThesisProposal.pdf
  ✓ Loaded 23 pages

Total documents loaded: 23


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='OBSERVING THE BEHAVIOUR OF EXISTING STRUCTURES \n \n \n \n \n \n \n \nA Thesis Proposal \nZEP WRITING SERVICES \n \n \n \n \n \n \n \n \nDecember 2021.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='Table of Contents \nChapter One .....................

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 23 documents into 58 chunks

Example chunk:
Content: OBSERVING THE BEHAVIOUR OF EXISTING STRUCTURES 
 
 
 
 
 
 
 
A Thesis Proposal 
ZEP WRITING SERVICES 
 
 
 
 
 
 
 
 
December 2021....
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='OBSERVING THE BEHAVIOUR OF EXISTING STRUCTURES \n \n \n \n \n \n \n \nA Thesis Proposal \nZEP WRITING SERVICES \n \n \n \n \n \n \n \n \nDecember 2021.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='Table of Contents \nChapter One .....................

# Embedding and Vector StoreDB

In [8]:
import pandas as pd  #Organizing data
from sentence_transformers import SentenceTransformer  #embedding model - Creating embeddings
import chromadb #Vector database
from chromadb.config import Settings
import uuid #Generating unique IDs
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity #Comparing vectors

/Users/nithiii/Projects/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 16:12:54.256000 17868 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #model from huggingface
        """
        Initialize the EmbeddingManager with a specified model name.
        Args:
            model_name (str): The name of the sentence transformer model to use for creating embeddings.
        """
        self.model_name = model_name
        self.model = None
        self.load_model()
    
    def load_model(self):
        """
        Load the sentence transformer model.
        """
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully.")

        except Exception as e:
            print(f"Error loading model: {e}")
            raise


    def create_embeddings(self, texts: List[str]) -> List[List[float]]:
        """
        Create embeddings for a list of texts.
        Args:
            texts (List[str]): A list of text strings to create embeddings for.
        Returns:
            List[List[float]]: A list of embeddings corresponding to the input texts.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        print(f"Creating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print("Embeddings created successfully.")
        return embeddings.tolist()

#initialize the EmbeddingManager
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11025.53it/s]


Model loaded successfully.


# Vectore Store

In [10]:
class VectorStore:
    """Manages document embeddings and their storage in a vector database."""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore with a ChromaDB client and collection.

        Args:
            collection_name (str): The name of the collection: chromadb
            persist_directory (str): The directory where the vector store will be persisted.
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_vector_store()

    def initialize_vector_store(self):
        """
        Initialize the ChromaDB client and collection.
        """
        try:
            #create the persist directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of document embeddings for RAG."}
            )
            print(f"Vector store initialized at '{self.persist_directory}' with collection '{self.collection_name}'.")
            print(f"Existing documents : {self.collection.count()} documents in the collection.")
            print("ChromaDB client initialized successfully.")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    #def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    def add_documents(self, documents: List[Any], embeddings: List[Any]):
        """Add documents and their embeddings to the vector store."""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(
                embedding.tolist() if hasattr(embedding, "tolist") else embedding
            )

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list,
            )
            print(f"Successfully added {len(documents)} documents.")
            print(f"Total documents: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise
        """Add documents and their embeddings to the vector store."""
        if len(documents) != len(embeddings):
            raise ValueError(
                "The number of documents must match the number of embeddings."
            )

        print(f"Adding {len(documents)} documents to the vector store.")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(
                embedding.tolist() if hasattr(embedding, "tolist") else embedding
            )

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list,
            )
            print(f"Successfully added {len(documents)} documents.")
            print(f"Total documents: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise
        """  Add documents and their embeddings to the VectorStore.
        Args:
            documents (List[Any]): A list of documents to add.
            embeddings (List[Any]): A list of embeddings corresponding to the documents.
        """       
        
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        print(f"Adding {len(documents)} documents to the vector store.")


        # prepare data for ChromaDB
        ids =[]
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content) if hasattr(doc, 'page_content') else 0
            metadatas.append(metadata)

            #Document content
            documents_texts.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist() if hasattr(embedding, "tolist") else embedding)

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise
Vectorstore=VectorStore()
Vectorstore



Vector store initialized at '../data/vector_store' with collection 'pdf_documents'.
Existing documents : 0 documents in the collection.
ChromaDB client initialized successfully.


In [11]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='OBSERVING THE BEHAVIOUR OF EXISTING STRUCTURES \n \n \n \n \n \n \n \nA Thesis Proposal \nZEP WRITING SERVICES \n \n \n \n \n \n \n \n \nDecember 2021.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-01-31T19:54:30+01:00', 'author': 'HP', 'moddate': '2023-01-31T19:54:30+01:00', 'source': '/Users/nithiii/Projects/data/pdf_files/SampleThesisProposal.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'SampleThesisProposal.pdf', 'file_type': 'pdf'}, page_content='Table of Contents \nChapter One .....................

In [14]:
#convert the chunks into embeddings
texts=[doc.page_content for doc in chunks]
#generate embeddings for the chunks and add them to the vector store
embeddings=embedding_manager.create_embeddings(texts)
#store in the vector database
Vectorstore.add_documents(chunks, embeddings) 

Creating embeddings for 58 texts.
Embeddings created successfully.
Successfully added 58 documents.
Total documents: 232
Adding 58 documents to the vector store.
Successfully added 58 documents.
Total documents: 290
Adding 58 documents to the vector store.
Successfully added 58 documents to the vector store.
Total documents in the collection: 348


# Retriever pipeline from VectorStore

In [24]:
class RAGRetriever:
    """Handles query based retrieval of documents from the vector store using embeddings."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAGRetriever with a vector store and an embedding manager.
        Args:
            vector_store (VectorStore): The vector store instance for document retrieval.
            embedding_manager (EmbeddingManager): The embedding manager instance for creating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold:float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on a query.
        Args:
            query (str): The input query string.
            top_k (int): The number of top documents to retrieve.
        Returns:
            List[Dict[str, Any]]: A list of retrieved documents with their metadata.
        """
        print(f"Retrieving documents for query: '{query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Create embedding for the query
        query_embedding = self.embedding_manager.create_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            #process results
            retrieved_docs = []
            if results['documents'] and results["documents"][0]:
                documents = results['documents'][0]
                metsdatas = results['documents'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents,metadata, distances)):
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print('No documets found')
                return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(Vectorstore, embedding_manager)

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("Numerical Analysis")

Retrieving documents for query: 'Numerical Analysis
Top K: 5, Score threshold: 0.0
Creating embeddings for 1 texts.
Embeddings created successfully.
Error during retrieval: 'list' object has no attribute 'tolist'


[]

# Integration vector Context pipeline with LLM output

In [27]:
#simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


False

In [28]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [29]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [30]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Please set your GROQ_API_KEY environment variable to use the LLM.


In [31]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework
Top K: 5, Score threshold: 0.0
Creating embeddings for 1 texts.
Embeddings created successfully.
Error during retrieval: 'list' object has no attribute 'tolist'


[]

# Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from getpass import getpass
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    groq_api_key = getpass("Enter your Groq API key: ")

groq_api_key = (os.getenv("GROQ_API_KEY") or "").strip()

if not groq_api_key:
    groq_api_key = getpass("Enter your Groq API key: ").strip()

if not groq_api_key:
    raise ValueError("A non-empty Groq API key is required.")

llm = ChatGroq(
    api_key=groq_api_key,
    model="gemma2-9b-it",
    temperature=0.1,
    max_tokens=1024,
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

# Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])